# 🏋️ Workout Efficacy Prediction
## Machine Learning Project — Houssem
### Objective: Classify workout intensity level (Low / Medium / High / Very High) from physiological and training data

**Models compared:** Decision Tree, Random Forest, KNN, SVM, Logistic Regression  
**Best model:** Random Forest (~82% accuracy)  
**Target:** `Intensity_Bin` — derived from actual `Calories_Burned` per session using quartile-based binning

> ⚠️ Note: The original dataset contains a column `Burns_Calories_Bin` which was NOT used.  
> That column is derived from an exercise reference table (calories per 30 min) and has no meaningful  
> correlation with actual session calories burned. We create a proper target from `Calories_Burned` instead.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print("✅ Imports successful")

## 2. Load Dataset

In [ ]:
df = pd.read_csv('df_clean.csv')
print(f'Shape: {df.shape}')
df.head(3)

## 3. Target Engineering
### Why we don't use `Burns_Calories_Bin`

The existing `Burns_Calories_Bin` column comes from an exercise reference table  
(calories burned per 30 minutes of a specific exercise). It has near-zero correlation  
with the actual calories burned during a session:

| Bin | Min session cal | Max session cal | Mean |
|-----|----------------|----------------|------|
| Low | 363 | **2341** | 1218 |
| High | 382 | **2296** | 1268 |
| Very High | 330 | **2332** | 1201 |

All bins have the same mean — they completely overlap. A model trained on this target  
would not be learning anything meaningful.

### Our approach: Quartile-based binning of `Calories_Burned`

In [ ]:
# Check actual Calories_Burned distribution
print("Calories_Burned statistics:")
print(df['Calories_Burned'].describe().round(2))
print()

# Create quartile-based bins
q1 = df['Calories_Burned'].quantile(0.25)
q2 = df['Calories_Burned'].quantile(0.50)
q3 = df['Calories_Burned'].quantile(0.75)

print(f'Q1 (25%): {q1:.0f} kcal')
print(f'Q2 (50%): {q2:.0f} kcal')
print(f'Q3 (75%): {q3:.0f} kcal')
print()

df['Intensity_Bin'] = pd.cut(
    df['Calories_Burned'],
    bins=[0, q1, q2, q3, 9999],
    labels=['Low', 'Medium', 'High', 'Very High']
)

target = 'Intensity_Bin'

print("Target distribution:")
print(df[target].value_counts().sort_index())
print()
print("Mean calories per bin:")
print(df.groupby(target)['Calories_Burned'].mean().round(0))

### 3.1 Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df[target].value_counts().sort_index()
colors = ['#4C9BE8', '#2A9D8F', '#F4A261', '#E76F51']
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.2)
axes[0].set_title('Intensity Bin Distribution', fontweight='bold')
axes[0].set_xlabel('Intensity Level')
axes[0].set_ylabel('Count')
for i, (idx, val) in enumerate(counts.items()):
    axes[0].text(i, val + 20, str(val), ha='center', fontweight='bold')

# Box plot of calories per bin
order = ['Low', 'Medium', 'High', 'Very High']
df.boxplot(column='Calories_Burned', by=target,
           ax=axes[1], patch_artist=True)
axes[1].set_title('Calories Burned per Intensity Bin', fontweight='bold')
axes[1].set_xlabel('Intensity Level')
axes[1].set_ylabel('Calories Burned (kcal)')
plt.suptitle('')

plt.tight_layout()
plt.show()

## 4. Feature Selection & EDA

In [ ]:
feature_cols = [
    'Age', 'Gender', 'Weight (kg)', 'Height (m)',
    'Max_BPM', 'Avg_BPM', 'Resting_BPM',
    'Session_Duration (hours)', 'Workout_Type',
    'Fat_Percentage', 'Water_Intake (liters)',
    'Workout_Frequency (days/week)', 'Experience_Level', 'BMI'
]

df_uc = df[feature_cols + [target]].copy().dropna()
print(f'Working dataset shape: {df_uc.shape}')
print(f'Missing values: {df_uc.isnull().sum().sum()}')

### 4.1 Feature Correlations with Target

In [ ]:
# Check which features actually correlate with target
from sklearn.preprocessing import LabelEncoder

df_corr = df_uc.copy()
le_tmp = LabelEncoder()
df_corr['Gender']       = le_tmp.fit_transform(df_corr['Gender'])
df_corr['Workout_Type'] = le_tmp.fit_transform(df_corr['Workout_Type'])
df_corr[target]         = le_tmp.fit_transform(df_corr[target])

corr = df_corr.corr()[target].drop(target).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
colors = ['#4C9BE8' if v > 0 else '#E76F51' for v in corr.values]
plt.barh(corr.index, corr.values, color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with Intensity_Bin', fontweight='bold')
plt.xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

print("Top positive correlations:")
print(corr.head(5).round(3))
print()
print("Top negative correlations:")
print(corr.tail(5).round(3))

### 4.2 Key Feature Distributions by Intensity

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Distributions by Intensity Level', fontsize=13, fontweight='bold')

key_features = ['Session_Duration (hours)', 'Experience_Level',
                'Workout_Frequency (days/week)', 'Avg_BPM', 'BMI', 'Age']
order = ['Low', 'Medium', 'High', 'Very High']
palette = {'Low': '#4C9BE8', 'Medium': '#2A9D8F', 'High': '#F4A261', 'Very High': '#E76F51'}

for ax, feat in zip(axes.flat, key_features):
    sns.boxplot(data=df_uc, x=target, y=feat, order=order,
                palette=palette, ax=ax)
    ax.set_title(feat, fontweight='bold', fontsize=10)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## 5. Preprocessing

In [ ]:
# Encode categoricals
le_gender  = LabelEncoder()
le_workout = LabelEncoder()
le_target  = LabelEncoder()

df_uc['Gender']       = le_gender.fit_transform(df_uc['Gender'].astype(str))
df_uc['Workout_Type'] = le_workout.fit_transform(df_uc['Workout_Type'].astype(str))
df_uc[target]         = le_target.fit_transform(df_uc[target].astype(str))

print('Target classes:', list(le_target.classes_))
print('Gender encoding:', dict(zip(['Female','Male'], le_gender.transform(['Female','Male']))))
print('Workout encoding:', dict(zip(le_workout.classes_, le_workout.transform(le_workout.classes_))))

In [ ]:
X = df_uc.drop(target, axis=1)
y = df_uc[target]

print(f'Features: {X.shape[1]} | Samples: {X.shape[0]}')
print(f'Missing in X: {X.isnull().sum().sum()}')

# Scale
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

## 6. Train All 5 Models

In [ ]:
models = {
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'SVM':                 SVC(kernel='rbf', random_state=42, probability=True),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
}

trained = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained[name] = model
    print(f'✅ {name} trained')

## 7. Evaluation

In [ ]:
results = []

for name, model in trained.items():
    y_pred = model.predict(X_test)
    cv     = cross_val_score(model, X_scaled, y, cv=5, scoring='f1_weighted')
    results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision': round(precision_score(y_test, y_pred, average='weighted') * 100, 2),
        'Recall':    round(recall_score(y_test, y_pred, average='weighted') * 100, 2),
        'F1 Score':  round(f1_score(y_test, y_pred, average='weighted') * 100, 2),
        'CV F1':     round(cv.mean() * 100, 2),
    })

results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False)
print(results_df.to_string(index=False))

In [ ]:
# Bar chart comparison
ax = results_df.set_index('Model')[['Accuracy','Precision','Recall','F1 Score']].plot(
    kind='bar', figsize=(12, 5), colormap='Set2'
)
plt.title('Model Comparison — Classification Metrics', fontweight='bold')
plt.ylabel('Score (%)')
plt.xticks(rotation=15, ha='right')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 8. Best Model Analysis

In [ ]:
best_name  = results_df.iloc[0]['Model']
best_model = trained[best_name]
y_pred_best = best_model.predict(X_test)

print(f'Best model: {best_name}')
print()
print(classification_report(y_test, y_pred_best, target_names=le_target.classes_))

In [ ]:
# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_, ax=axes[0])
axes[0].set_title(f'Confusion Matrix — {best_name}', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Feature importance
rf_model   = trained['Random Forest']
importance = pd.Series(rf_model.feature_importances_, index=X.columns)
importance.sort_values(ascending=True).plot(
    kind='barh', figsize=(10, 4), color='#4C9BE8', ax=axes[1]
)
axes[1].set_title('Feature Importance — Random Forest', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

### 8.1 Cross-Validation

In [ ]:
cv_results = {}
for name, model in trained.items():
    cv = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
    cv_results[name] = cv

cv_df = pd.DataFrame(cv_results)

plt.figure(figsize=(10, 5))
cv_df.boxplot()
plt.title('5-Fold Cross-Validation Accuracy per Model', fontweight='bold')
plt.ylabel('Accuracy')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

print('CV Mean Accuracy:')
for name, scores in cv_results.items():
    print(f'  {name}: {scores.mean():.4f} ± {scores.std():.4f}')

## 9. Export Models

In [ ]:
import joblib, os

os.makedirs('models/houssem', exist_ok=True)

joblib.dump(best_model,  'models/houssem/workout_model.pkl')
joblib.dump(scaler,      'models/houssem/workout_scaler.pkl')
joblib.dump(le_target,   'models/houssem/workout_le_target.pkl')

print(f'✅ Best model: {best_name}')
print(f'✅ Saved to models/houssem/')
print()

# Save quartile thresholds for Streamlit page
thresholds = {
    'q1': float(df['Calories_Burned'].quantile(0.25)),
    'q2': float(df['Calories_Burned'].quantile(0.50)),
    'q3': float(df['Calories_Burned'].quantile(0.75)),
}
import json
with open('models/houssem/thresholds.json', 'w') as f:
    json.dump(thresholds, f)

print(f'Thresholds: {thresholds}')
print(f'Target classes: {list(le_target.classes_)}')